# LLM-AGR Reproduction for Colab — Amazon-book & Yelp

**Run all cells top-to-bottom at the start of every session.**  
The grid cell skips already-completed runs, so sessions are safe to resume.




In [32]:
!rm -rf /content/LLM-AGR
!git clone https://github.com/mertrodop/CS_555_Project.git /content/LLM-AGR

!ls -la /content/LLM-AGR
!find /content/LLM-AGR -maxdepth 2 -name "main.py"

Cloning into '/content/LLM-AGR'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 107 (delta 36), reused 106 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 212.47 KiB | 1.17 MiB/s, done.
Resolving deltas: 100% (36/36), done.
total 68
drwxr-xr-x 7 root root  4096 May 26 16:21 .
drwxr-xr-x 1 root root  4096 May 26 16:21 ..
-rw-r--r-- 1 root root 15073 May 26 16:21 aggregate.py
drwxr-xr-x 4 root root  4096 May 26 16:21 config
drwxr-xr-x 8 root root  4096 May 26 16:21 .git
-rw-r--r-- 1 root root   349 May 26 16:21 .gitignore
-rw-r--r-- 1 root root  8105 May 26 16:21 llm_agr_repro.ipynb
drwxr-xr-x 3 root root  4096 May 26 16:21 load_data
-rw-r--r-- 1 root root   594 May 26 16:21 main.py
drwxr-xr-x 4 root root  4096 May 26 16:21 models
-rw-r--r-- 1 root root  1409 May 26 16:21 README.md
-rw-r--r-- 1 root root    59 May 26 16:21 requirements.txt
drwxr-xr-x 3 root root

In [33]:
import shutil

shutil.rmtree("/content/LLM-AGR/logs", ignore_errors=True)
shutil.rmtree("/content/drive/MyDrive/LLM-AGR-results/logs", ignore_errors=True)

print("Old logs cleared.")

Old logs cleared.


In [34]:
# ── Cell 2: Copy data from Drive to local /content (fast SSD) ─────────────────
import shutil, os

DATA_DRIVE = '/content/drive/MyDrive/LLM-AGR-data'
DATA_LOCAL = '/content/LLM-AGR/data'
os.makedirs(DATA_LOCAL, exist_ok=True)

for ds in ['amazon', 'yelp']:
    src = f'{DATA_DRIVE}/{ds}'
    dst = f'{DATA_LOCAL}/{ds}'
    assert os.path.exists(src), f"Missing {src} — upload data to Drive first"
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f'Copied {ds} data to {dst}')
    else:
        print(f'{ds} data already present — skipping copy')

Copied amazon data to /content/LLM-AGR/data/amazon
Copied yelp data to /content/LLM-AGR/data/yelp


In [35]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
import os, torch

# Standard packages (pyyaml, scipy, tqdm, pandas)
os.system('pip install -r /content/LLM-AGR/requirements.txt -q')

# PyG packages need version-specific wheel URL — auto-detect from Colab's runtime
torch_ver = torch.__version__.split('+')[0]
cuda_ver  = torch.version.cuda.replace('.', '')
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html'
print(f'Installing torch_sparse + torch_scatter from:\n  {pyg_url}')
os.system(f'pip install torch_sparse torch_scatter -f {pyg_url} -q')

# Verify
import torch_sparse, torch_scatter
print('All dependencies installed successfully')

Installing torch_sparse + torch_scatter from:
  https://data.pyg.org/whl/torch-2.10.0+cu128.html
All dependencies installed successfully


In [36]:
# ── Cell 4: Restore completed logs from Drive (makes grid resumable) ───────────
import shutil, os

DRIVE_LOGS = '/content/drive/MyDrive/LLM-AGR-results/logs'
LOCAL_LOGS = '/content/LLM-AGR/logs'
os.makedirs(LOCAL_LOGS, exist_ok=True)

if os.path.exists(DRIVE_LOGS):
    shutil.copytree(DRIVE_LOGS, LOCAL_LOGS, dirs_exist_ok=True)
    n = sum(1 for _, _, fs in os.walk(LOCAL_LOGS) for f in fs if f.endswith('.log'))
    print(f'Restored {n} log(s) from Drive — those runs will be skipped')
else:
    print('No existing logs on Drive — starting fresh')

No existing logs on Drive — starting fresh


In [37]:
%%writefile /content/LLM-AGR/run_grid.sh
#!/bin/bash
set -e
cd /content/LLM-AGR
DRIVE_LOGS="/content/drive/MyDrive/LLM-AGR-results/logs"

for model in lightgcn lightgcn_agr sgl sgl_agr simgcl simgcl_agr bigcf bigcf_agr; do
  for dataset in amazon yelp; do
    for seed in 0 1 2 3 4; do
      LOG="logs/${dataset}/${model}_seed${seed}.log"
      mkdir -p "logs/${dataset}"
      if [ -f "$LOG" ]; then
        echo "[SKIP] $LOG already exists"
        continue
      fi
      echo "====== ${model} | ${dataset} | seed=${seed} ======"
      python main.py --model $model --dataset $dataset --seed $seed \
        2>&1 | tee "$LOG"
      # Immediately sync this log to Drive so a session reset doesn't lose it
      mkdir -p "${DRIVE_LOGS}/${dataset}"
      cp "$LOG" "${DRIVE_LOGS}/${dataset}/"
    done
  done
done
echo "Grid complete."

Writing /content/LLM-AGR/run_grid.sh


In [38]:
# ── Cell 5b: Run the grid (this cell will run for several hours) ───────────────
# Each completed log is synced to Drive immediately, so you can interrupt
# and resume in a new session without losing progress.
os.makedirs('/content/drive/MyDrive/LLM-AGR-results/logs', exist_ok=True)
os.system('bash /content/LLM-AGR/run_grid.sh')

0

In [ ]:
# ── Cell 6: Final sync — run any time to push all logs & results to Drive ──────
import shutil, os

DRIVE_OUT = '/content/drive/MyDrive/LLM-AGR-results'
os.makedirs(DRIVE_OUT, exist_ok=True)

for folder in ['logs', 'results', 'checkpoint']:
    src = f'/content/LLM-AGR/{folder}'
    dst = f'{DRIVE_OUT}/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Synced {folder}/ → Drive')

# Count completed runs
logs_dir = f'{DRIVE_OUT}/logs'
n = sum(1 for _, _, fs in os.walk(logs_dir) for f in fs if f.endswith('.log'))
print(f'\n{n}/80 runs completed so far')

## After All 80 Runs — Run Aggregation

Once all logs are collected, run the `aggregate.py` script (committed to your repo) to produce:
- `results/amazon_book.csv`, `results/yelp.csv`
- `results/table3_repro.md` (mean ± std, significance markers)
- `results/paper_vs_ours.md` (gap to paper numbers)
- `results/SUMMARY.md`

In [40]:
# ── Cell 7 (run after all 80 logs exist): Aggregate results ───────────────────
import os, shutil

DRIVE_OUT = '/content/drive/MyDrive/LLM-AGR-results'
os.chdir('/content/LLM-AGR')
os.makedirs('results', exist_ok=True)
os.system('python aggregate.py')
# Sync final results to Drive
shutil.copytree('results', f'{DRIVE_OUT}/results', dirs_exist_ok=True)
print('Aggregation complete — results synced to Drive')

Aggregation complete — results synced to Drive


In [24]:
import os, subprocess

REPO_DIR = "/content/LLM-AGR"

print("Current repo folder exists:", os.path.exists(REPO_DIR))

if os.path.exists(REPO_DIR):
    print("\nRemote URL:")
    os.system(f"cd {REPO_DIR} && git remote -v")

    print("\nCurrent branch:")
    os.system(f"cd {REPO_DIR} && git branch --show-current")

    print("\nLatest commit:")
    os.system(f"cd {REPO_DIR} && git log -1 --oneline")

    print("\nFiles in repo root:")
    os.system(f"ls -la {REPO_DIR}")

    print("\nFind main.py:")
    os.system(f"find {REPO_DIR} -name 'main.py'")

Current repo folder exists: True

Remote URL:

Current branch:

Latest commit:

Files in repo root:

Find main.py:
